In [1]:
%%writefile increment_arrayelements.cu

#include <stdio.h>
#include "cuda_runtime.h"
#include "device_launch_parameters.h"

#define N 5

__global__ void increment(int *a)
{
    int idx = blockIdx.x * blockDim.x + threadIdx.x;
    if (idx < N)
    {
        a[idx] = a[idx] + 1;
    }
}

int main()
{
    int host_a[N] = {1, 2, 3, 4, 5};

    // device pointer
    int *device_a;

    // allocate memory on device
    cudaMalloc((void **)&device_a, N * sizeof(int));

    // copy input array from host to device
    cudaMemcpy(device_a, host_a, N * sizeof(int), cudaMemcpyHostToDevice);

    // launch kernel: one block, N threads (one per element)
    increment<<<1, N>>>(device_a);
    cudaDeviceSynchronize();

    // copy result back from device to host (into the same array)
    cudaMemcpy(host_a, device_a, N * sizeof(int), cudaMemcpyDeviceToHost);

    // display the result
    printf("Incremented: ");
    for (int i = 0; i < N; i++)
        printf("%d ", host_a[i]);
    printf("\n");

    // free memory
    cudaFree(device_a);

    return 0;
}

Writing increment_arrayelements.cu


In [2]:
!nvcc increment_arrayelements.cu -o increment_arrayelements

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).


In [3]:
!./increment_arrayelements

Incremented: 2 3 4 5 6 
